In [ ]:
%pip install transformers

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install scikit-learn

In [ ]:
import csv

from transformers import pipeline
# import plotly as plt
from tqdm import tqdm
import plotly.express as px
import pandas as pd
from sklearn.decomposition import PCA


In [ ]:
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

In [ ]:
# Answering a question by garo
result = clf("assault", ['crime', 'safety', 'climate'], multi_label=True)
result

In [ ]:
seq = "hello world!"
candidate_labels = ["greeting", "food"]
result = clf(seq, candidate_labels, multi_label=True)
result

In [1]:
import requests
import io

# filename = '/kaggle/input/datasets/jeromegv/subject-terms-txt/subject_terms.txt'
# subject_terms = []
# with open(filename, 'r', encoding='utf-8') as file:
#     subject_terms = file.readlines()
# subject_terms = [term.strip() for term in subject_terms]

github_url = "https://raw.githubusercontent.com/civic-dashboard/civic-dashboard-web/375tagginguireal/subject_terms.txt"
response = requests.get(github_url)

if response.status_code == 200:
    subject_terms = [term.strip() for term in response.text.splitlines() if term.strip()]
    print(f"Successfully loaded {len(subject_terms)} terms from GitHub.")
else:
    print(f"Failed to fetch file. Status code: {response.status_code}")

subject_terms[:5]

Successfully loaded 3683 terms from GitHub.


['20 minute makeover', '25 year club', '3 1 1', '3 rs', '311']

In [ ]:
# Those are the list of tentative tags we are trying against the list of normalized subject terms
# This is tracked by this github ticket: https://github.com/civic-dashboard/civic-dashboard-web/issues/139
categories = [
    "Accessibility & Equity",
    "Animals & Wildlife",
    "Arts, Culture & Events",
    "Budget, Taxes & Finance",
    "Business & Economy",
    "Children & Youth",
    "Construction & Infrastructure",
    "Environment & Climate",
    "Governance & City Hall",
    "Housing & Homelessness",
    "Licensing & Permits",
    "Noise & Nuisances",
    "Parks & Public Spaces",
    "Public Health & Safety",
    "Seniors & Aging",
    "Sports & Recreation",
    "Transportation & Mobility",
    "Urban Planning & Zoning",
    "Waste Management",
    "Water & Sewers"
]

In [ ]:
seq = "3-1-1"
candidate_labels = categories
result = clf(subject_terms[:100], candidate_labels, multi_label=True)
result

In [ ]:
dict(zip(result[0]['labels'], result[0]['scores']))

In [ ]:
# Compute similarities

filename = 'similarities.csv'
with open(filename, 'w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['subject_term'] + list(categories))
    writer.writeheader()

    batch_index = 0
    batch_size = 50
    progress = tqdm(total=len(subject_terms))
    while batch_index * batch_size <= len(subject_terms):
    # for term in tqdm(subject_terms[:100]):
        curr_term_batch = subject_terms[batch_index * batch_size:(batch_index + 1) * batch_size]
        results = clf(curr_term_batch, categories, multi_label=True)
        output = []
        for result in results:
            row = dict(zip(result['labels'], result['scores']))
            row['subject_term'] = result['sequence']
            output.append(row)
        
        writer.writerows(output)
        progress.update(batch_size)
        batch_index += 1
    progress.close()

In [ ]:
def reduce_dimensionality(df: pd.DataFrame, num_dimensions: int) -> pd.DataFrame:
    pca = PCA(n_components=num_dimensions)
    pca.fit(df)
    reduced_df = pd.DataFrame(pca.transform(df))
    return reduced_df

In [ ]:
df = pd.read_csv('similarities.csv')
df['max_val'] = df.drop(columns=['subject_term']).max(axis=1)
df['max_cat'] = df.drop(columns=['subject_term']).idxmax(axis=1)
# df

In [ ]:
# Add subject term frequency
from collections import Counter
import numpy as np

# subject_term_counter = Counter()
# with open("/kaggle/input/datasets/jeromegv/subject-terms-txt/subject_terms.txt", 'r', encoding='utf-8') as file:
#     for line in file:
#         term = line.strip()
#         subject_term_counter[term] += 1

# Since we already fetched the list from GitHub in Cell 8, we use it directly:
subject_term_counter = Counter(subject_terms)

df['frequency'] = 0
frequencies = []
for i, row in df.iterrows():
    frequencies.append(subject_term_counter[row['subject_term']])
df['frequency'] = frequencies
assert df['frequency'].all()

df['log_frequency'] = np.log(df['frequency'])
df

In [ ]:
reduced_df = pd.DataFrame(df)
reduced_df.drop(columns=['subject_term', 'max_val', 'max_cat'], inplace=True)
reduced_df = reduce_dimensionality(reduced_df, 3)
reduced_df.insert(0, 'subject_term', pd.Series(df['subject_term']))
reduced_df

In [ ]:
fig = px.scatter_3d(reduced_df, x=0, y=1, z=2, hover_name='subject_term')
fig.show()

In [ ]:
fig = px.scatter_matrix(df, dimensions=categories[:5], width=1600, height=800, hover_name='subject_term')
fig.show()

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42)
projections = tsne.fit_transform(df.drop(columns=['subject_term', 'max_val', 'max_cat', 'frequency', 'log_frequency']))
projections = pd.DataFrame(projections)
projections.insert(0, 'subject_term', pd.Series(df['subject_term']))
projections.insert(1, 'max_category', pd.Series(df['max_cat']))
projections.insert(2, 'max_value', pd.Series(df['max_val']).map(lambda x: round(x, 2)))
projections.insert(3, 'frequency', pd.Series(df['frequency']))
projections.insert(4, 'log_frequency', pd.Series(df['log_frequency'].map(lambda x: round(x, 2))))

In [ ]:
fig = px.scatter(projections, x=0, y=1, hover_name='subject_term', color='max_category', hover_data=['max_category', 'max_value', 'frequency'],
                 size='log_frequency', width=1600, height=800)
fig.show()

In [ ]:
# Display top candidate subject terms for each category
threshold = 0.85
top_terms = df[df['max_val'] >= threshold]
top_terms

In [ ]:
top_terms.to_csv("top_terms.csv")

In [ ]:
df.to_csv("all_terms.csv")

In [ ]:
# Generate histograms of similarities per category
import plotly.graph_objects as go
import numpy as np

# fig = px.histogram(
#     df, 
#     x='housing',
#     width=500,
#     height=500,
#     nbins=20
# )
fig = go.Figure()
for cat in categories:
    fig.add_trace(go.Histogram(
        x=df[cat],
        xbins=dict(
            start=0.0,
            end=1.0,
            size=0.05
        ),
        name=cat.capitalize()
    ))
fig.update_layout(barmode='overlay', width=500)
fig.update_traces(opacity=0.75)
fig.show()

In [ ]:
fig = px.histogram(
    df,
    x='max_val',
    color='max_cat',
    marginal='rug',
    hover_name='max_cat',
    width=1000,
    height=750,
    nbins=10,
    title='Most similar category of agenda items\' subject terms',
)
fig.show()

In [ ]:
for cat in categories:
    fig = px.histogram(
        df,
        x=cat,
        marginal='rug',
        nbins=20,
        width=500,
        height=500,
        # color='max_cat'
    )
    fig.show()

In [ ]:
# FIX: Use 'max_val' instead of a specific category name
# This shows you the distribution of your model's "Best Guesses"
counts, bins = np.histogram(df['max_val'], bins=[i / 100 for i in range(0, 101, 5)])
bins = .5 * (bins[:-1] + bins[1:])
fig = px.bar(x=bins, y=counts, title="Distribution of Top Confidence Scores")
fig.show()

In [ ]:
bins, counts

In [ ]:
histogram_table = []
for cat in categories:
    counts, bins = np.histogram(df[cat], bins=[i / 100 for i in range(0, 101, 5)])
    histogram_table.append(counts.tolist())
    # bins = .5 * (bins[:-1] + bins[1:])
    # fig = px.bar(x=bins, y=counts)
    # fig.show()
histogram_table = pd.DataFrame(
    histogram_table,
    columns=[
        "0.00 - 0.05",
        "0.05 - 0.10",
        "0.10 - 0.15",
        "0.15 - 0.20",
        "0.20 - 0.25",
        "0.25 - 0.30",
        "0.30 - 0.35",
        "0.35 - 0.40",
        "0.40 - 0.45",
        "0.45 - 0.50",
        "0.50 - 0.55",
        "0.55 - 0.60",
        "0.60 - 0.65",
        "0.65 - 0.70",
        "0.70 - 0.75",
        "0.75 - 0.80",
        "0.80 - 0.85",
        "0.85 - 0.90",
        "0.90 - 0.95",
        "0.95 - 1.00"
    ],
    index=categories
)
histogram_table

In [ ]:
histogram_table.to_csv('histogram_table.csv')

In [ ]:
sorted(categories)

---

In [ ]:
from collections import Counter

# count_terms = Counter()
# with open('/kaggle/input/datasets/jeromegv/subject-terms-txt/subject_terms.txt', 'r', encoding='utf-8') as file:
#     for line in file:
#         line = line.strip()
#         count_terms[line] += 1

# Using the pre-loaded data
count_terms = Counter(subject_terms)

In [ ]:
sorted_terms = sorted(count_terms, reverse=True, key=lambda x: count_terms[x])

In [ ]:
for term in sorted_terms:
    print(term, count_terms[term])

In [ ]:
len(count_terms), df.shape

In [ ]:
df.iloc[4000:4005]['subject_term'], sorted(count_terms)[4000:4005]

In [ ]:
df['count'] = [count_terms[term] for term in sorted(count_terms)]
df.iloc[4000:4005]

In [ ]:
count_terms['smell']

In [ ]:
df.to_csv('all_terms.csv')